## Density Analysis

This analysis uses the wifi connections to infer foot traffic. Here we simply cross reference addresses with their proximity to water bottle filling stations. We do this using latitude and longitude coordinates.

Our end deliverable is a ranking of 3 addresses that have a high distance from a filler station with a large amount of people in that range.




### Import and Visualize Datasets

In [1]:
import pandas as pd
density = pd.read_csv('data/updated_densitymap_hourly_cleaned.csv')
bot_fill = pd.read_csv('data/bottle_fillers.csv')
bot_fill_cord = pd.read_csv('data/bottle_filler_coordinates.csv')
events = pd.read_csv('data/high_volume_events.csv', header=1)

In [2]:
# Remove addresses that are nan
density = density.dropna(subset=['latitude', 'longitude'])
bottle_fill = bot_fill.dropna(subset=['Address A', 'Address B'])

In [3]:
density.head()

,date,hour,building_floor,building_desc,latitude,longitude,building_type,density_cnt,capacity,dc_ratio
0,2021-03-31,1,4,209 BAY STATE ROAD,42.3507,-71.1028,Residential,1,8,0.13
1,2021-03-31,1,2,200-202 BAY STATE ROAD,42.3503,-71.1026,Residential,1,12,0.08
2,2021-03-31,1,2,184 BAY STATE ROAD,42.3503,-71.1019,Residential,1,6,0.17
3,2021-03-31,1,3,190 BAY STATE ROAD,42.3503,-71.1021,Residential,1,9,0.11
4,2021-03-31,1,3,161 BAY STATE ROAD,42.3506,-71.1009,Residential,1,5,0.20


In [4]:
bot_fill.head()

,Address A,Address B,Floor,Description,Accessibility,Handicap Accessible,Space@Bu room,Quantity,Date Installed,Type,Walk-Thru Notes,Completed
0,100,Ashford St,First,"Hallway across from vending machines, across f...",Public,Yes,191,1.0,NaN,Combo Bottle Filler/Drinking Fountain,two additional drinking fountain locations in ...,1/5/2024
1,120,Ashford St,First,"Kitchenette, Room 116A",Restricted,Yes,116A,1.0,NaN,Office-Style Water Cooler,NaN,1/5/2024
2,120,Ashford St,Second,"Kitchenette, Room 222",Restricted,Yes,222,1.0,NaN,Office-Style Water Cooler,NaN,1/5/2024
3,8,St. Mary's Street,Basement,Hallway by bathrooms,NaN,NaN,B81,1.0,Dec-23,Combo Bottle Filler/Drinking Fountain,NaN,NaN
4,8,St. Mary's Street,First,Hallway by electrical room,NaN,NaN,181,1.0,NaN,NaN,NaN,NaN


In [5]:
bot_fill_cord.head()

,Unnamed: 0,address,latitudes,longitudes
0,0,"300 Babcock St, Boston, MA",42.354249,-71.121241
1,1,"820 Commonwealth Ave, Boston, MA",42.350493,-71.112035
2,2,"36 Cummington Mall, Boston, MA",42.348504,-71.102190
3,3,"64 Cummington Mall, Boston, MA",42.348728,-71.103538
4,4,"725 Commonwealth Ave, Boston, MA",42.350609,-71.109893


In [6]:
events.head(20)

,Room Name,Address,Floor,Unnamed: 3,Current,Unnamed: 5,Notes
0,Agganis Arena Burke Club Room,925 Commonwealth Avenue,2nd Floor,NaN,0,NaN,"casual catered events, high liklihood of using..."
1,Agganis Arena Friends of Hockey Lounge,925 Commonwealth Avenue,1st Floor,NaN,0,NaN,"casual catered events, high liklihood of using..."
2,Case Center Eilberg Lounge,285 Babcock Street,3rd Floor,NaN,??,NaN,use of the space is limited; there are fillers...
3,College of Arts & Sciences Classrooms,675-725 Commonwealth Avenue,NaN,NaN,NaN,complete FY23,There are a bunch already down the hallways; 4...
4,College of Communication Lounge,640 Commowealth Avenue,1st Floor,NaN,1,NaN,hallway by room 105
5,College of General Studies Gilbane Lounge,871 Commonwealth Avenue,1st Floor,NaN,NaN,complete FY23,one on each floor
6,College of General Studies Katzenberg Lounge,871 Commonwealth Avenue,4th Floor,NaN,NaN,complete FY23,NaN
7,College of General Studies Sleeper Auditorium,871 Commonwealth Avenue,1st Floor,NaN,NaN,complete FY23,NaN
8,Dahod Family Alumni Center,225 Bay State Road,1st Floor,NaN,??,NaN,There's one in the space btw Castle and Admiss...
9,George Sherman Union Alley,775 Commonwealth Avenue,Basement,NaN,1,NaN,The space is used heavily for dance/performanc...


### Drop all residential buildings

In [7]:
# Get average capacity and dc_ratio for all time at a given location
filtered_density = density
filtered_density = filtered_density.drop(filtered_density[filtered_density['building_type'] == 'Residential'].index).reset_index(drop=True)
print(f"Dropping the residential buildings reduced the dataset to {len(filtered_density)} / {len(density)}")

Dropping the residential buildings reduced the dataset to 4474731 / 12170420


### Only consider times from A to B

In [8]:
import numpy as np 

time_A = 8 # 8am
time_B = 20 # 8pm

inds_before_A = filtered_density['hour'] < time_A
inds_after_B = filtered_density['hour'] > time_B


indicies_to_drop = np.nonzero(np.logical_or(inds_before_A, inds_after_B))[0]
filtered_density = filtered_density.drop(indicies_to_drop, axis=0)


In [9]:
filtered_density.head()

,date,hour,building_floor,building_desc,latitude,longitude,building_type,density_cnt,capacity,dc_ratio
405,2021-03-31,8,3,881 COMMONWEALTH AVENUE,42.3514,-71.1153,Administrative,5,70,0.07
406,2021-03-31,8,5,808 COMMONWEALTH AVENUE,42.3501,-71.1117,Academic,6,176,0.03
407,2021-03-31,8,1,808 COMMONWEALTH AVENUE,42.3501,-71.1117,Academic,24,320,0.08
408,2021-03-31,8,2,808 COMMONWEALTH AVENUE,42.3501,-71.1117,Academic,12,317,0.04
409,2021-03-31,8,3,808 COMMONWEALTH AVENUE,42.3501,-71.1117,Academic,1,185,0.01


### Now we want to take the treat each building as a single data point
We do this by grouping the data

In [10]:
average_counts = filtered_density.groupby(['building_desc'])[['latitude', 'longitude', 'density_cnt', 'dc_ratio']].mean().reset_index()
average_floor_counts = filtered_density.groupby(['building_desc', 'building_floor'])[['latitude', 'longitude', 'density_cnt', 'dc_ratio']].mean().reset_index()

In [11]:
average_counts.head()

,building_desc,latitude,longitude,density_cnt,dc_ratio
0,1 UNIVERSITY ROAD,42.3512,-71.1100,36.698204,0.138214
1,10 LENOX STREET,42.3487,-71.1099,6.042315,0.185805
2,100 ASHFORD STREET,42.3546,-71.1236,17.998309,0.195150
3,100 BAY STATE ROAD,42.3498,-71.0979,26.072143,0.198703
4,1010 COMMONWEALTH AVENUE,42.3513,-71.1210,27.136059,0.320439


In [12]:
average_floor_counts.head()


,building_desc,building_floor,latitude,longitude,density_cnt,dc_ratio
0,1 UNIVERSITY ROAD,1,42.3512,-71.1100,60.441044,0.196502
1,1 UNIVERSITY ROAD,2,42.3512,-71.1100,21.977137,0.187538
2,1 UNIVERSITY ROAD,3,42.3512,-71.1100,6.966685,0.082629
3,1 UNIVERSITY ROAD,p,42.3512,-71.1100,42.588065,0.000000
4,10 LENOX STREET,1,42.3487,-71.1099,6.823363,0.145285


### Add a feature: Clossest distance to a water fountain

In [13]:
from utils import haversine

def find_clossest_water_fountain_dist(lat, lon, locations):
    p1 = (lat, lon)
    distances = [[haversine(p1, location), add] for location, add in locations]
    return min(distances)

### Add a feature: Density counts weighted by distance to filter

In [14]:
bot_fill_cord.head()
latitudes = bot_fill_cord.loc[:, 'latitudes'].tolist()
longitudes = bot_fill_cord.loc[:, 'longitudes'].tolist()
addresses = bot_fill_cord.loc[:, 'address'].tolist()
locations = list(zip(zip(latitudes, longitudes), addresses))

WALKING_TIME_MIN_PER_KM = 12.25

apply_clossest_dist = lambda row: find_clossest_water_fountain_dist(row['latitude'], row['longitude'], locations)[0]
apply_clossest_address = lambda row: find_clossest_water_fountain_dist(row['latitude'], row['longitude'], locations)[1]
average_counts['nearest_filler'] = average_counts.apply(apply_clossest_dist, axis=1)
average_counts['nearest_filler_address'] = average_counts.apply(apply_clossest_address, axis=1)
average_counts['filler_walking_dist'] = average_counts['nearest_filler'] * WALKING_TIME_MIN_PER_KM
average_counts['score'] = average_counts['density_cnt'] * average_counts['nearest_filler']
average_counts.head(10)

,building_desc,latitude,longitude,density_cnt,dc_ratio,nearest_filler,nearest_filler_address,filler_walking_dist,score
0,1 UNIVERSITY ROAD,42.3512,-71.1100,36.698204,0.138214,0.031135,"1 University Rd, Boston, MA",0.381409,1.142615
1,10 LENOX STREET,42.3487,-71.1099,6.042315,0.185805,0.201382,"808 Commonwealth Ave , Boston, MA",2.466932,1.216815
2,100 ASHFORD STREET,42.3546,-71.1236,17.998309,0.195150,0.055380,"100 Ashford St, Boston, MA",0.678406,0.996748
3,100 BAY STATE ROAD,42.3498,-71.0979,26.072143,0.198703,0.041613,"565 Commonwealth Ave, Boston, MA",0.509754,1.084930
4,1010 COMMONWEALTH AVENUE,42.3513,-71.1210,27.136059,0.320439,0.080199,"985 Commonwealth Ave, Boston, MA",0.982436,2.176282
5,110-112 CUMMINGTON MALL,42.3491,-71.1055,6.222999,0.100011,0.050259,"96 Cummington Mall, Boston, MA",0.615676,0.312763
6,111 CUMMINGTON MALL,42.3491,-71.1040,17.676116,0.024311,0.044953,"68 Cummington Mall, Boston, MA",0.550675,0.794595
7,116 BAY STATE ROAD,42.3500,-71.0985,2.061205,0.324892,0.011462,"100 Bay State Road, Boston, MA",0.140410,0.023626
8,118 BAY STATE ROAD,42.3500,-71.0986,2.233843,0.319432,0.017051,"100 Bay State Road, Boston, MA",0.208881,0.038090
9,120 ASHFORD STREET,42.3544,-71.1223,14.736887,0.130912,0.005611,"120 Ashford St, Boston, MA",0.068735,0.082689


### Now, we can rank the locations based on the densicy_count * walking distance

In [15]:
## Compute the top k locations with high density * distance to nearest filter
k = 10

filtered_avg_counts = average_counts[average_counts['filler_walking_dist'] > 1]
top_locations = filtered_avg_counts.sort_values(by='score', ascending=False)
top_locations.head(10)
# top_locations.iloc[:100, :].to_csv('top_capacity_times_distance_to_water.csv')


,building_desc,latitude,longitude,density_cnt,dc_ratio,nearest_filler,nearest_filler_address,filler_walking_dist,score
21,163-179 AMORY STREET,42.3502,-71.1136,43.939831,0.412706,0.132244,"855 Commonwealth Ave, Boston, MA",1.619986,5.810767
87,949 COMMONWEALTH AVENUE,42.3519,-71.1187,41.967803,0.920422,0.101689,"33 Harry Agganis Way, Boston, MA",1.245685,4.267645
67,750 COMMONWEALTH AVENUE,42.3498,-71.1080,20.955958,0.099109,0.143225,"767 Commonwealth Ave, Boston, MA",1.754507,3.001418
65,736 - 738 COMMONWEALTH AVENUE,42.3497,-71.1072,22.424723,0.638827,0.105317,"8 St. Mary's Street, Boston, MA",1.290137,2.361712
35,278 BABCOCK STREET,42.3535,-71.1219,25.471229,0.000000,0.091964,"277 Babcock St, Boston, MA",1.126555,2.342427
51,619 MEMORIAL DRIVE,42.3532,-71.1078,8.510821,0.197255,0.229131,"771 Commonwealth Ave, Boston, MA",2.806858,1.950095
1,10 LENOX STREET,42.3487,-71.1099,6.042315,0.185805,0.201382,"808 Commonwealth Ave , Boston, MA",2.466932,1.216815
81,910 COMMONWEALTH AVENUE,42.3508,-71.1166,10.487884,0.139071,0.089353,"915 Commonwealth Ave, Boston, MA",1.094575,0.937125
44,53 BAY STATE ROAD,42.3502,-71.0951,10.139024,0.595979,0.082457,"610 Beacon St, Boston, MA",1.010104,0.836039
18,15 ST. MARY'S STREET,42.3494,-71.1075,6.772048,0.041712,0.116412,"8 St. Mary's Street, Boston, MA",1.426051,0.788350


### Repeat with a similar metric, but instead just rank based on density and use a threshold walking distance

In [16]:
# Compute the top k locations with the highest density given a threshold of walking distance

filtered_avg_counts1 = average_counts[average_counts['filler_walking_dist'] > 1]
highest_capacity = filtered_avg_counts1.sort_values(by='density_cnt', ascending=False)
highest_capacity = highest_capacity.iloc[:, [0, 3, 6, 7]]
highest_capacity.head(k)
# highest_capacity.iloc[:100, :].to_csv('data/highest_capacity_without_water.csv')


,building_desc,density_cnt,nearest_filler_address,filler_walking_dist
21,163-179 AMORY STREET,43.939831,"855 Commonwealth Ave, Boston, MA",1.619986
87,949 COMMONWEALTH AVENUE,41.967803,"33 Harry Agganis Way, Boston, MA",1.245685
35,278 BABCOCK STREET,25.471229,"277 Babcock St, Boston, MA",1.126555
65,736 - 738 COMMONWEALTH AVENUE,22.424723,"8 St. Mary's Street, Boston, MA",1.290137
67,750 COMMONWEALTH AVENUE,20.955958,"767 Commonwealth Ave, Boston, MA",1.754507
81,910 COMMONWEALTH AVENUE,10.487884,"915 Commonwealth Ave, Boston, MA",1.094575
44,53 BAY STATE ROAD,10.139024,"610 Beacon St, Boston, MA",1.010104
51,619 MEMORIAL DRIVE,8.510821,"771 Commonwealth Ave, Boston, MA",2.806858
18,15 ST. MARY'S STREET,6.772048,"8 St. Mary's Street, Boston, MA",1.426051
1,10 LENOX STREET,6.042315,"808 Commonwealth Ave , Boston, MA",2.466932


### Helper Functions

In [17]:
import re

def average_address_if_range(address):
    ''' Takes in a range of addresses and returns the average address
        Ex: 20 - 30 Buswell St -> 25 Buswell St
    '''
    regex = r"(\d+)\s*-\s*(\d+)(.*)"
    parts = re.fullmatch(regex, address)
    if parts == None:
        return address

    start = int(parts.group(1))
    end = int(parts.group(2))
    street = parts.group(3)

    new = (start + end) // 2
    return str(new) + street